# Milestone 2 — Audio Dataset Analysis with PyTorch
Answers to all 7 questions using the `genres_stems` train split.

In [1]:
import sys, os

# ── torch lives in E:/tmp_pip (too large for C drive); all other packages
#    are installed in the venv on F: drive (no C-drive impact).
#    Append E:/tmp_pip at the END so venv packages take priority. ────────
_pip_path = "E:/tmp_pip"
if _pip_path not in sys.path:
    sys.path.append(_pip_path)   # END of path — venv packages win

# Sanity-check imports
import numpy, pandas, librosa, soundfile, tqdm, torch

print("✓ All libraries available")
print(f"  numpy:     {numpy.__version__}  from {numpy.__file__[:50]}")
print(f"  pandas:    {pandas.__version__}")
print(f"  librosa:   {librosa.__version__}")
print(f"  soundfile: {soundfile.__version__}")
print(f"  torch:     {torch.__version__}")
print(f"  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not available — using CPU'}")


✓ All libraries available
  numpy:     2.4.2  from f:\Downloads\jan-2026-dl-gen-ai-project\.venv\Lib\
  pandas:    3.0.1
  librosa:   0.11.0
  soundfile: 0.13.1
  torch:     2.10.0+cpu
  GPU: not available — using CPU


In [2]:
import os, random, sys
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import torch
from tqdm import tqdm

# ── GPU check ─────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

# ── Config (mirrors Milestone 1) ──────────────────────────────────────────
DATA_ROOT = r"f:\Downloads\jan-2026-dl-gen-ai-project\messy_mashup\genres_stems"
SR        = 22050
DURATION  = 5.0
TOP_DB    = 20

random.seed(67)
np.random.seed(67)

GENRES    = sorted([g for g in os.listdir(DATA_ROOT)
                    if os.path.isdir(os.path.join(DATA_ROOT, g))])
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
STEMS     = {
    'drums.wav':  'drums',
    'vocals.wav': 'vocals',
    'bass.wav':   'bass',
    'other.wav':  'other',
}

print(f"Genres found: {GENRES}")


Using device: cpu
Genres found: ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']


In [3]:
# ── Replicate Milestone-1 train/val split (identical parameters) ──────────
def build_dataset(root_dir, val_split=0.17, seed=42):
    train_dataset = {g: {k: [] for k in STEM_KEYS} for g in GENRES}
    val_dataset   = {g: {k: [] for k in STEM_KEYS} for g in GENRES}
    rng = random.Random(seed)

    for genre in GENRES:
        genre_path = os.path.join(root_dir, genre)
        songs = sorted(os.listdir(genre_path))
        valid_songs = []

        for song in songs:
            song_path  = os.path.join(genre_path, song)
            stem_files = []
            for stem_file in STEMS:
                fpath = os.path.join(song_path, stem_file)
                if not os.path.exists(fpath):
                    break
                stem_files.append(fpath)
            if len(stem_files) == 4:
                valid_songs.append(song_path)

        rng.shuffle(valid_songs)
        split_idx   = int(len(valid_songs) * (1 - val_split))
        train_songs = valid_songs[:split_idx]
        val_songs   = valid_songs[split_idx:]

        for s in train_songs:
            for stem_file in STEMS:
                train_dataset[genre][STEMS[stem_file]].append(
                    os.path.join(s, stem_file))

        for s in val_songs:
            for stem_file in STEMS:
                val_dataset[genre][STEMS[stem_file]].append(
                    os.path.join(s, stem_file))

    return train_dataset, val_dataset

tr, val = build_dataset(DATA_ROOT)

total_train = sum(len(tr[g][s]) for g in GENRES for s in STEM_KEYS)
print(f"Total train files : {total_train}")
print(f"Jazz train drums  : {len(tr['jazz']['drums'])}")


Total train files : 3320
Jazz train drums  : 83


In [4]:
# ══════════════════════════════════════════════════════════════════════════
# Q1 — Mean duration (seconds) of Jazz genre stems in the train dataset
# ══════════════════════════════════════════════════════════════════════════
jazz_files = [fp for stem in STEM_KEYS for fp in tr['jazz'][stem]]
print(f"Total Jazz train files: {len(jazz_files)}")

durations = []
for fp in tqdm(jazz_files, desc="Jazz durations"):
    try:
        info = sf.info(fp)          # fast – no decoding
        durations.append(info.duration)
    except Exception:
        try:
            y, s = librosa.load(fp, sr=None)
            durations.append(len(y) / s)
        except Exception:
            pass   # truly unreadable – skip

dur_tensor = torch.tensor(durations, dtype=torch.float32)
mean_dur   = dur_tensor.mean().item()

print(f"\n--- Q1 ---")
print(f"Mean duration of Jazz stems in train dataset: {mean_dur:.4f} seconds")


Total Jazz train files: 332


Jazz durations: 100%|██████████| 332/332 [00:00<00:00, 1714.37it/s]


--- Q1 ---
Mean duration of Jazz stems in train dataset: 30.0346 seconds


In [13]:

# ══════════════════════════════════════════════════════════════════════════
# Q2 — Unique sample rates present in the ENTIRE dataset
#       (scan ALL .wav files recursively under messy_mashup root;
#        includes genres_stems, mashups, and ESC-50-master/audio)
# ══════════════════════════════════════════════════════════════════════════
DATASET_ROOT = r"f:\Downloads\jan-2026-dl-gen-ai-project\messy_mashup"

sample_rates = set()
for dirpath, _, filenames in os.walk(DATASET_ROOT):
    for fname in filenames:
        if fname.endswith('.wav'):
            fp = os.path.join(dirpath, fname)
            try:
                info = sf.info(fp)
                sample_rates.add(info.samplerate)
            except Exception:
                pass

unique_srs = sorted(sample_rates)
print(f"\n--- Q2 ---")
print(f"Unique sample rates in dataset: {unique_srs}")



--- Q2 ---
Unique sample rates in dataset: [22050, 44100]


In [6]:
# ══════════════════════════════════════════════════════════════════════════
# Q3 — Corrupted OR zero-byte audio files in the train dataset
#       "Corrupted" = zero-byte (size==0) OR file that cannot be decoded
# ══════════════════════════════════════════════════════════════════════════
train_files = [fp for g in GENRES for s in STEM_KEYS for fp in tr[g][s]]
print(f"Total train files to scan: {len(train_files)}")

corrupted = []
for fp in tqdm(train_files, desc="Checking corrupted/zero-byte"):
    size = os.path.getsize(fp)
    if size == 0:
        corrupted.append((fp, "zero-byte"))
        continue
    try:
        sf.info(fp)                  # lightweight header check
    except Exception:
        try:
            librosa.load(fp, sr=None, duration=0.1)  # try partial decode
        except Exception:
            corrupted.append((fp, "unreadable"))

print(f"\n--- Q3 ---")
print(f"Corrupted or zero-byte files in train dataset: {len(corrupted)}")
if corrupted:
    for fp, reason in corrupted[:10]:
        print(f"  [{reason}] {fp}")


Total train files to scan: 3320


Checking corrupted/zero-byte: 100%|██████████| 3320/3320 [00:00<00:00, 6899.83it/s]


--- Q3 ---
Corrupted or zero-byte files in train dataset: 0


In [7]:
# ══════════════════════════════════════════════════════════════════════════
# Q4 — Average peak amplitude (dB) for vocal stems in the train dataset
#       Peak amplitude (dB) = 20 * log10( max|signal| )
#       Files with max amplitude == 0 are skipped (log undefined).
# ══════════════════════════════════════════════════════════════════════════
vocal_files = [fp for g in GENRES for fp in tr[g]['vocals']]
print(f"Total vocal train files: {len(vocal_files)}")

peak_db_list = []
for fp in tqdm(vocal_files, desc="Vocal peak amplitudes"):
    try:
        y, _ = librosa.load(fp, sr=SR, mono=True)
        peak = float(np.max(np.abs(y)))
        if peak > 0:
            peak_db = 20.0 * np.log10(peak)
            peak_db_list.append(peak_db)
    except Exception:
        pass

peak_tensor     = torch.tensor(peak_db_list, dtype=torch.float32)
avg_peak_db     = peak_tensor.mean().item()

print(f"\n--- Q4 ---")
print(f"Average peak amplitude for vocal stems (train): {avg_peak_db:.4f} dB")


Total vocal train files: 830


Vocal peak amplitudes:   0%|          | 0/830 [00:00<?, ?it/s]f:\Downloads\jan-2026-dl-gen-ai-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Vocal peak amplitudes: 100%|██████████| 830/830 [00:36<00:00, 22.72it/s]


--- Q4 ---
Average peak amplitude for vocal stems (train): -12.5442 dB


In [8]:
# ══════════════════════════════════════════════════════════════════════════
# Q5 & Q6 — Spectral Centroid per genre in the train dataset
#
#  Q5: Mean spectral centroid for 'blues' genre
#  Q6: Genre with the HIGHEST mean spectral centroid
#
#  Method: librosa.feature.spectral_centroid → mean over all frames and
#          files per genre (all 4 stems combined).
# ══════════════════════════════════════════════════════════════════════════
genre_centroid_means = {}

for genre in tqdm(GENRES, desc="Genres (spectral centroid)"):
    genre_files = [fp for s in STEM_KEYS for fp in tr[genre][s]]
    all_centroids = []

    for fp in tqdm(genre_files, desc=f"  {genre}", leave=False):
        try:
            y, _ = librosa.load(fp, sr=SR, mono=True)
            if len(y) == 0 or np.max(np.abs(y)) == 0:
                continue
            sc = librosa.feature.spectral_centroid(y=y, sr=SR)  # shape (1, T)
            all_centroids.append(float(np.mean(sc)))
        except Exception:
            pass

    if all_centroids:
        t = torch.tensor(all_centroids, dtype=torch.float32)
        genre_centroid_means[genre] = t.mean().item()
    else:
        genre_centroid_means[genre] = float('nan')

print(f"\n--- Q5 ---")
blues_sc = genre_centroid_means.get('blues', float('nan'))
print(f"Mean spectral centroid for 'blues' (train): {blues_sc:.4f} Hz")

print(f"\n--- Q6 ---")
best_genre = max(genre_centroid_means, key=lambda g: genre_centroid_means[g])
print(f"Genre with highest mean spectral centroid: {best_genre} ({genre_centroid_means[best_genre]:.4f} Hz)")
print("\nAll genres:")
for g, v in sorted(genre_centroid_means.items(), key=lambda x: -x[1]):
    print(f"  {g:12s}: {v:.4f} Hz")


Genres (spectral centroid): 100%|██████████| 10/10 [03:57<00:00, 23.75s/it]


--- Q5 ---
Mean spectral centroid for 'blues' (train): 2099.9365 Hz

--- Q6 ---
Genre with highest mean spectral centroid: metal (2519.9675 Hz)

All genres:
  metal       : 2519.9675 Hz
  pop         : 2367.1108 Hz
  disco       : 2363.4368 Hz
  reggae      : 2257.3342 Hz
  jazz        : 2240.6978 Hz
  hiphop      : 2232.7012 Hz
  rock        : 2159.6147 Hz
  blues       : 2099.9365 Hz
  country     : 2086.1243 Hz
  classical   : 1433.9180 Hz


In [9]:
# ══════════════════════════════════════════════════════════════════════════
# Q7 — Stem audio files in the train dataset that contain silence
#       in the FIRST 0.5 seconds
#
#  Method: load audio → take first 0.5 s → use librosa.effects.split
#  with TOP_DB=20 on that sub-clip.
#  A file "contains silence in first 0.5 s" if:
#    - the sub-clip is fully silent (no non-silent intervals), OR
#    - the first non-silent interval doesn't start at sample 0,    OR
#    - there is any gap (silent region) within the 0.5-s window.
# ══════════════════════════════════════════════════════════════════════════
half_sec_samples = int(0.5 * SR)   # 11025 samples at 22050 Hz

silence_in_first_05 = 0
train_files_all = [fp for g in GENRES for s in STEM_KEYS for fp in tr[g][s]]
print(f"Scanning {len(train_files_all)} train files …")

for fp in tqdm(train_files_all, desc="Silence in first 0.5s"):
    try:
        y, _ = librosa.load(fp, sr=SR, mono=True)
        y_half = y[:half_sec_samples]                # first 0.5 s
        if len(y_half) == 0:
            silence_in_first_05 += 1                 # empty = silent
            continue

        intervals = librosa.effects.split(y_half, top_db=TOP_DB)

        if len(intervals) == 0:
            # entire sub-clip is silent
            silence_in_first_05 += 1
        else:
            # Check: does the first interval start at 0 AND cover fully?
            fully_covered = (
                len(intervals) == 1
                and intervals[0][0] == 0
                and intervals[0][1] == len(y_half)
            )
            if not fully_covered:
                silence_in_first_05 += 1
    except Exception:
        pass   # unreadable files are not counted

print(f"\n--- Q7 ---")
print(f"Train files with silence in first 0.5 seconds: {silence_in_first_05}")


Scanning 3320 train files …


Silence in first 0.5s: 100%|██████████| 3320/3320 [01:47<00:00, 30.91it/s]


--- Q7 ---
Train files with silence in first 0.5 seconds: 1159


---
## Decision Tree Classifier — Genre Classification
Answers to Q8–Q13 using the provided Decision Tree code.

In [11]:
import os
import numpy as np
import pandas as pd
import librosa
import matplotlib
matplotlib.use('Agg')   # non-interactive backend for VS Code
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report, accuracy_score

# --- 1. Setup ---
ROOT       = r'f:\Downloads\jan-2026-dl-gen-ai-project\messy_mashup'
STEMS_PATH = os.path.join(ROOT, 'genres_stems')
GENRES_DT  = ["blues", "classical", "country", "disco", "hiphop",
               "jazz", "metal", "pop", "reggae", "rock"]

def extract_features(song_path):
    """Load 10 s of other.wav and return 4 hand-crafted features."""
    y, sr = librosa.load(os.path.join(song_path, 'other.wav'), sr=22050, duration=10)
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    spec_cent = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    zcr       = np.mean(librosa.feature.zero_crossing_rate(y))
    rolloff   = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
    # librosa ≥0.10 returns tempo as 0-d or 1-d array — normalise to scalar
    tempo_val = float(np.atleast_1d(tempo)[0])
    return [tempo_val, float(spec_cent), float(zcr), float(rolloff)]

# --- 2. Data Preparation & Stratified Split ---
data = []
for g in GENRES_DT:
    gp    = os.path.join(STEMS_PATH, g)
    songs = [s for s in os.listdir(gp) if os.path.isdir(os.path.join(gp, s))]
    for s in songs[:50]:            # 50 songs per genre for speed
        data.append({'path': os.path.join(gp, s), 'genre': g})

df_dt = pd.DataFrame(data)
print(f"Total songs in dataset: {len(df_dt)}")

train_df, val_df = train_test_split(df_dt, test_size=0.2,
                                    stratify=df_dt['genre'], random_state=42)
print(f"Train: {len(train_df)}  Val: {len(val_df)}")

# --- 3. Feature Extraction ---
print("Extracting train features …")
X_train = np.array([extract_features(p) for p in train_df['path']])
y_train = train_df['genre'].values

print("Extracting val features …")
X_val = np.array([extract_features(p) for p in val_df['path']])
y_val = val_df['genre'].values

# --- 4. Model Training ---
clf = DecisionTreeClassifier(max_depth=5, random_state=42)
clf.fit(X_train, y_train)

# --- 5. Predictions & Metrics ---
y_pred    = clf.predict(X_val)
macro_f1  = f1_score(y_val, y_pred, average='macro')
cm        = confusion_matrix(y_val, y_pred, labels=GENRES_DT)
cr        = classification_report(y_val, y_pred, labels=GENRES_DT)
accuracy  = accuracy_score(y_val, y_pred)

print(f"\nValidation Macro F1 Score: {macro_f1:.4f}\n")
print(f"Model Accuracy: {accuracy:.4f}\n")
print("Detailed Classification Report:")
print(cr)


Total songs in dataset: 500
Train: 400  Val: 100
Extracting train features …
Extracting val features …

Validation Macro F1 Score: 0.1523

Model Accuracy: 0.2000

Detailed Classification Report:
              precision    recall  f1-score   support

       blues       0.43      0.30      0.35        10
   classical       0.00      0.00      0.00        10
     country       0.12      0.20      0.15        10
       disco       0.21      0.70      0.33        10
      hiphop       0.00      0.00      0.00        10
        jazz       0.08      0.10      0.09        10
       metal       0.38      0.30      0.33        10
         pop       0.00      0.00      0.00        10
      reggae       0.20      0.40      0.27        10
        rock       0.00      0.00      0.00        10

    accuracy                           0.20       100
   macro avg       0.14      0.20      0.15       100
weighted avg       0.14      0.20      0.15       100



f:\Downloads\jan-2026-dl-gen-ai-project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
f:\Downloads\jan-2026-dl-gen-ai-project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
f:\Downloads\jan-2026-dl-gen-ai-project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.c

In [12]:
# ══════════════════════════════════════════════════════════════════════════
# Confusion Matrix Visualisation + Per-Genre TP, TN, FP, FN
# ══════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=GENRES_DT, yticklabels=GENRES_DT, ax=ax)
ax.set_title('Confusion Matrix — Decision Tree (max_depth=5)', fontsize=14)
ax.set_xlabel('Predicted Genre', fontsize=12)
ax.set_ylabel('True Genre', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120)
plt.show()
print("Confusion matrix saved as confusion_matrix.png\n")

# --- Per-Genre TP, TN, FP, FN ---
print(f"{'Genre':<12} {'TP':>5} {'TN':>6} {'FP':>5} {'FN':>5}")
print("-" * 35)

rows = {}
for i, genre in enumerate(GENRES_DT):
    TP = cm[i, i]
    FP = cm[:, i].sum() - TP      # predicted as this genre but wrong
    FN = cm[i, :].sum() - TP      # actual this genre but predicted wrong
    TN = cm.sum() - TP - FP - FN
    rows[genre] = {'TP': TP, 'TN': TN, 'FP': FP, 'FN': FN}
    print(f"{genre:<12} {TP:>5} {TN:>6} {FP:>5} {FN:>5}")

# --- Summary ---
max_tp_genre  = max(rows, key=lambda g: rows[g]['TP'])
min_fn_genre  = min(rows, key=lambda g: rows[g]['FN'])

print(f"\nGenre with highest True Positives  : {max_tp_genre}  (TP={rows[max_tp_genre]['TP']})")
print(f"Genre with lowest  False Negatives : {min_fn_genre}  (FN={rows[min_fn_genre]['FN']})")


Confusion matrix saved as confusion_matrix.png

Genre           TP     TN    FP    FN
-----------------------------------
blues            3     86     4     7
classical        0     89     1    10
country          2     76    14     8
disco            7     64    26     3
hiphop           0     89     1    10
jazz             1     79    11     9
metal            3     85     5     7
pop              0     88     2    10
reggae           4     74    16     6
rock             0     90     0    10

Genre with highest True Positives  : disco  (TP=7)
Genre with lowest  False Negatives : disco  (FN=3)


C:\Users\shiba\AppData\Local\Temp\ipykernel_27332\2558694454.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
